<div style="border-top: 1px solid rgba(128, 128, 128, 0.45); padding-top: 10px;"><span style="font-size: 11px; letter-spacing: 0.12em; text-transform: uppercase; opacity: 0.65;">Walkthrough &middot; about 2 min</span></div>

## <span style="font-family: Newsreader, Georgia, serif; font-size: 34px; font-weight: 400;">3. Data Download &amp; Preparation</span>

<div style="font-size: 19px; line-height: 1.5; max-width: 44em;">The data, the official CUB-200-2011 release, was downloaded and verified against its published checksum.</div>

### 3.1 Source

The repository's own downloader fetches a [third-party mirror](https://github.com/IBM/concept_transformer/blob/84c9c30108a6249e017d580c4120ea54af0064f7/data/cub2011parts.py#L29) and [checks no checksum](https://github.com/IBM/concept_transformer/blob/84c9c30108a6249e017d580c4120ea54af0064f7/data/cub2011parts.py#L276). We took the official archive from [CaltechDATA](https://data.caltech.edu/records/65de6-vp158) instead, matched it against the MD5 that CaltechDATA publishes, and decoded every image once before training. The cell shows that audit and re-hashes the annotation files on disk.

In [ ]:
REHASH_ANNOTATIONS = True  # False shows the saved audit only

import hashlib
from html import escape

source_audit = json.loads((SETUP_LOGS / "cub-download.json").read_text())
assert source_audit["status"] == "passed" and source_audit["archive_md5_matches_published"]
audited_images, audited_classes = source_audit["images"], source_audit["classes"]
audit_rows = [
    ("Source", source_audit["checksum_source_url"]),
    ("Archive", f"{source_audit['archive_bytes'] / 1e9:.2f} GB, MD5 {source_audit['archive_hashes']['md5']}, "
                "matches the published value"),
    ("Images", f"{audited_images['fully_decoded']:,} of {audited_images['referenced']:,} decoded in full"),
    ("Species", f"{audited_classes['count']}, "
                + ("each present in both official splits" if audited_classes["all_classes_in_both_splits"]
                   else "NOT all present in both official splits")),
    ("Official split", f"{source_audit['split_counts']['train']:,} training / "
                       f"{source_audit['split_counts']['test']:,} test images"),
    ("Annotations", f"{source_audit['attributes']['rows']:,} image–attribute labels, "
                    f"{source_audit['parts']['rows']:,} part locations"),
    ("Audit finished", source_audit["audit_completed_utc"][:19].replace("T", " ") + " UTC"),
]
if Path(source_audit["loader_root"]) != DATA_DIR:  # the audit ran on another copy of the same files
    audit_rows.append(("Audited copy", f"{source_audit['dataset_path']}; this session reads {DATA_DIR / 'CUB_200_2011'}"))
if REHASH_ANNOTATIONS:
    changed = [name for name, expected in source_audit["annotation_sha256"].items()
               if hashlib.sha256((DATA_DIR / "CUB_200_2011" / name).read_bytes()).hexdigest() != expected]
    audit_rows.append(("Annotation files here",
                       f"all {len(source_audit['annotation_sha256'])} identical to the audited files (SHA-256)"
                       if not changed else "DIFFERENT from the audited files: " + ", ".join(changed)))
fact_table(audit_rows, title="Source data audit")
display(HTML(
    '<details style="font-size: 13.5px; margin-top: 6px;"><summary style="cursor: pointer; opacity: 0.8;">'
    f'All {len(source_audit["checks"])} audit checks</summary><ul>'
    + "".join(f"<li>{escape(check)}</li>" for check in source_audit["checks"]) + "</ul></details>"
))

<details><summary style="cursor: pointer; font-size: 15px;"><b>Backup:</b> download and audit commands</summary>

Run once from the repository root on HiPerGator. They are preparation, not cells for the call.

```bash
.venv/bin/python scripts/download_cub.py               # download, MD5 check, extract, copy attributes.txt inward
.venv/bin/python scripts/download_cub.py --check-only  # verify an existing copy without downloading
.venv/bin/python scripts/verify_cub.py                 # full audit, writes setup_logs/cub-download.json
```

Segmentation masks are not used by the model, so they are not downloaded.

</details>

### 3.2 Directory layout

`cvit_cub.py` passes `--data_dir` straight to the loader, which expects the **outer** `CUB_200_2011` folder and appends the inner one itself. The loader reads `attributes.txt` from the inner folder, but the archive extracts it to the outer one, so we copy it inward and keep the original.

<div style="max-width: 820px; font-size: 14px; line-height: 1.45; margin-top: 4px;">
<div style="border: 1px solid rgba(128, 128, 128, 0.45); border-top: 3px solid rgba(128, 128, 128, 0.45); padding: 8px 14px 10px 14px;">
<div style="font-size: 11px; letter-spacing: 0.12em; text-transform: uppercase; opacity: 0.65;">DATA_DIR &middot; passed as --data_dir</div>
<div style="font-family: Newsreader, Georgia, serif; font-size: 20px; margin: 2px 0 6px 0;"><code>CUB-200-2011/CUB_200_2011/</code></div>
<div style="border-top: 1px solid rgba(128, 128, 128, 0.25); padding: 5px 6px; background: rgba(190, 124, 28, 0.08);"><span style="display: inline-block; width: 44%; vertical-align: top; overflow-wrap: anywhere;"><code>attributes.txt</code></span><span style="display: inline-block; width: 54%; vertical-align: top; opacity: 0.8;">Where the archive extracts the attribute names. The loader never reads this copy.</span></div>
<div style="border: 1px solid rgba(128, 128, 128, 0.45); border-top: 3px solid rgba(128, 128, 128, 0.45); padding: 8px 14px 10px 14px; margin-top: 10px;">
<div style="font-size: 11px; letter-spacing: 0.12em; text-transform: uppercase; opacity: 0.65;">Inner folder &middot; the loader appends it to --data_dir</div>
<div style="font-family: Newsreader, Georgia, serif; font-size: 20px; margin: 2px 0 6px 0;"><code>CUB_200_2011/</code></div>
<div style="border-top: 1px solid rgba(128, 128, 128, 0.25); padding: 5px 6px; background: rgba(172, 84, 63, 0.07);"><span style="display: inline-block; width: 44%; vertical-align: top; overflow-wrap: anywhere;"><code>attributes.txt</code></span><span style="display: inline-block; width: 54%; vertical-align: top; opacity: 0.8;">Our byte-identical copy of the file above. The loader reads the attribute names only here.</span></div>
<div style="border-top: 1px solid rgba(128, 128, 128, 0.25); padding: 5px 6px;"><span style="display: inline-block; width: 44%; vertical-align: top; overflow-wrap: anywhere;"><code>cropped/</code></span><span style="display: inline-block; width: 54%; vertical-align: top; opacity: 0.8;">Bounding-box crops, made once by the repository code: the images the model sees.</span></div>
<div style="border-top: 1px solid rgba(128, 128, 128, 0.25); padding: 5px 6px;"><span style="display: inline-block; width: 44%; vertical-align: top; overflow-wrap: anywhere;"><code>images/</code></span><span style="display: inline-block; width: 54%; vertical-align: top; opacity: 0.8;">Source photos. Not fed to the model, but the loader checks that every one exists.</span></div>
<div style="border-top: 1px solid rgba(128, 128, 128, 0.25); padding: 5px 6px;"><span style="display: inline-block; width: 44%; vertical-align: top; overflow-wrap: anywhere;"><code>images.txt</code><br><code>classes.txt</code><br><code>image_class_labels.txt</code><br><code>train_test_split.txt</code><br><code>bounding_boxes.txt</code></span><span style="display: inline-block; width: 54%; vertical-align: top; opacity: 0.8;">Image paths, species names, labels, the official train/test split and the bird bounding boxes.</span></div>
<div style="border-top: 1px solid rgba(128, 128, 128, 0.25); padding: 5px 6px;"><span style="display: inline-block; width: 44%; vertical-align: top; overflow-wrap: anywhere;"><code>attributes/image_attribute_labels.txt</code></span><span style="display: inline-block; width: 54%; vertical-align: top; opacity: 0.8;">Which attributes each image shows: the source of the concept targets.</span></div>
<div style="border-top: 1px solid rgba(128, 128, 128, 0.25); padding: 5px 6px;"><span style="display: inline-block; width: 44%; vertical-align: top; overflow-wrap: anywhere;"><code>parts/part_locs.txt</code></span><span style="display: inline-block; width: 54%; vertical-align: top; opacity: 0.8;">Part locations, which place the spatial concepts on 16 &times; 16-pixel patches.</span></div>
</div>
</div>
<div style="font-size: 13.5px; line-height: 1.45; opacity: 0.8; margin-top: 6px;">Only the files the loader reads are shown. The live cell below prints the full tree from disk.</div>
</div>

<div style="border-top: 3px solid #ac543f; padding-top: 8px;"><span style="font-size: 11px; letter-spacing: 0.12em; text-transform: uppercase; font-weight: 600;">Live demo</span><span style="font-size: 13.5px; opacity: 0.7;"> &middot; read-only &middot; change <code>MAX_DEPTH</code> and re-run</span></div>

In [ ]:
# Live: the folder layout cvit_cub.py expects under --data_dir. Read-only.
MAX_DEPTH = 3  # folder levels shown below DATA_DIR
MAX_LISTED = 20  # folders with more entries than this are summarised, not listed

DATASET_DIR = DATA_DIR / "CUB_200_2011"
TREE_NOTES = {
    DATA_DIR / "attributes.txt": "as extracted from the archive",
    DATASET_DIR: "inner folder, appended by the loader",
    DATASET_DIR / "attributes.txt": "copy where the loader reads it",
    DATASET_DIR / "images": "source photos",
    DATASET_DIR / "cropped": "bounding-box crops made by the repository code",
}


def visible_entries(folder):
    return [entry for entry in folder.iterdir() if not entry.name.startswith(".")]


def print_tree(folder, prefix="", depth=0):
    entries = sorted(visible_entries(folder), key=lambda entry: (not entry.is_dir(), entry.name))
    for i, entry in enumerate(entries):
        last = i == len(entries) - 1
        notes = [TREE_NOTES[entry]] if entry in TREE_NOTES else []
        children = visible_entries(entry) if entry.is_dir() else []
        if len(children) > MAX_LISTED:
            kind = "folders" if all(child.is_dir() for child in children) else "entries"
            notes.append(f"{len(children)} {kind}, {sum(1 for _ in entry.rglob('*.jpg')):,} JPEGs")
        line = prefix + ("└── " if last else "├── ") + entry.name + ("/" if entry.is_dir() else "")
        print(f"{line:<52}← " + "; ".join(notes) if notes else line)
        if entry.is_dir() and depth + 1 < MAX_DEPTH and len(children) <= MAX_LISTED:
            print_tree(entry, prefix + ("    " if last else "│   "), depth + 1)


assert DATASET_DIR.is_dir(), f"Check DATA_DIR in the setup cell: {DATA_DIR}"
print(f"{DATA_DIR.name + '/':<52}← DATA_DIR, passed as --data_dir")
print_tree(DATA_DIR)

### 3.3 Preprocessing

Preprocessing has two parts. Steps 1–2 prepare files on disk once, before any training; they are done and need no repeating, apart from one check before each run. Steps 3–7 run automatically inside the repository's data loader every time an image is loaded, so nothing needs to be done for them.

**Before training: once, on disk**

| # | Step | Run by | Status |
| --- | --- | --- | --- |
| 1 | Crop each of the 11,788 photos to the bird's bounding box and save it in `cropped/` | the repository's [`_crop_images`](https://github.com/IBM/concept_transformer/blob/84c9c30108a6249e017d580c4120ea54af0064f7/data/cub2011parts.py#L285-L297), on the first data load; we then decoded every crop (`setup_logs/cub-cropping.json`) | Done. Before each training run, confirm all 11,788 crops exist: the repository skips cropping whenever the folder exists (Section 3.4) |
| 2 | Fix the train / validation / test split | Repository run: nothing; `setup()` draws it at load time. Disjoint run: `split_manifest.json`, generated once (Section 3.5) | Done |

**During training: automatic, for every image the loader reads**

| # | Step | Applied to | Repository code |
| --- | --- | --- | --- |
| 3 | Resize to 224 × 224 | all images | [`CUB2011Parts` transforms](https://github.com/IBM/concept_transformer/blob/84c9c30108a6249e017d580c4120ea54af0064f7/data/cub2011parts_datamodule.py#L22-L32) |
| 4 | Random horizontal flip (p = 0.5) and rotation within ±30° | training images only | [`train_transform`](https://github.com/IBM/concept_transformer/blob/84c9c30108a6249e017d580c4120ea54af0064f7/data/cub2011parts_datamodule.py#L22-L27) |
| 5 | Normalize with the ImageNet mean and standard deviation | all images | [both transforms](https://github.com/IBM/concept_transformer/blob/84c9c30108a6249e017d580c4120ea54af0064f7/data/cub2011parts_datamodule.py#L22-L32) |
| 6 | Move the annotated part keypoints through steps 3 and 4 with the image | all images | [`__getitem__`](https://github.com/IBM/concept_transformer/blob/84c9c30108a6249e017d580c4120ea54af0064f7/data/cub2011parts.py#L302-L348) |
| 7 | Build the concept targets from those moved keypoints: 13 global concepts per image, and 95 spatial concepts per patch of the 14 × 14 grid. These 108 are the attributes, out of 312, that the repository keeps | all images | [`_get_explanations`](https://github.com/IBM/concept_transformer/blob/84c9c30108a6249e017d580c4120ea54af0064f7/data/cub2011parts.py#L219-L252) |

Step 6 is why the spatial targets stay on the bird after a flip or rotation: the keypoints are transformed together with the pixels, and each patch's target is read from where the keypoint lands.

<div style="border-top: 3px solid #ac543f; padding-top: 8px;"><span style="font-size: 11px; letter-spacing: 0.12em; text-transform: uppercase; font-weight: 600;">Live demo</span><span style="font-size: 13.5px; opacity: 0.7;"> &middot; change <code>SAMPLE_INDEX</code> or <code>AUG_SEED</code> and re-run</span></div>

In [ ]:
# Live: one training image through the repository's preprocessing, with the concept targets it produces.
SAMPLE_INDEX = 0  # position in the official training set, 0 to len(train_set) - 1
AUG_SEED = 42  # seed for the random flip and rotation

import random
import torch
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle
from matplotlib.ticker import MaxNLocator
from PIL import Image
from data import CUB2011Parts, CUB2011Parts_dataset

torch.set_num_threads(CPU_THREADS)
DATASET_DIR = DATA_DIR / "CUB_200_2011"
assert (DATASET_DIR / "cropped").is_dir(), "No crop cache: the dataset would start writing crops. See 3.4."
data_module = CUB2011Parts(data_dir=str(DATA_DIR), batch_size=2, num_workers=0)  # holds both transforms
train_set = CUB2011Parts_dataset(root=str(DATA_DIR), train=True, crop=True, download=False)
assert 0 <= SAMPLE_INDEX < len(train_set), f"SAMPLE_INDEX must be below {len(train_set)}"
sample_meta = train_set.data.iloc[SAMPLE_INDEX]


def fetch(transform, seed=None):
    """The sample through one transform; a seed fixes the random flip and rotation, then the RNGs are restored."""
    saved = random.getstate(), np.random.get_state()
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)
    train_set.transform = transform
    try:
        return train_set[SAMPLE_INDEX]
    finally:
        random.setstate(saved[0])
        np.random.set_state(saved[1])


def to_rgb(image):
    """Undo the ImageNet normalization for display."""
    mean, std = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])
    return np.clip(image.permute(1, 2, 0).numpy() * std + mean, 0, 1)


def patch_targets(spatial):
    """(196, 95) targets to a 14 x 14 grid: which patches carry targets, and positives per patch."""
    known = torch.isfinite(spatial).any(dim=1).numpy().reshape(14, 14)
    positives = torch.nan_to_num(spatial, nan=0.0).sum(dim=1).numpy().reshape(14, 14)
    return known, positives


eval_image, eval_global, eval_spatial, class_target = fetch(data_module.test_transform)
aug_image, _, aug_spatial, _ = fetch(data_module.train_transform, seed=AUG_SEED)
with Image.open(DATASET_DIR / "images" / sample_meta.filepath) as photo:
    source_rgb = photo.convert("RGB")

eval_known, eval_positives = patch_targets(eval_spatial)
aug_known, aug_positives = patch_targets(aug_spatial)
cmap = LinearSegmentedColormap.from_list("targets", ["#d99a88", PALETTE["accent"]])
vmax = max(2.0, float(eval_positives.max()), float(aug_positives.max()))  # at least two colour steps
fig, axes = plt.subplots(1, 3, figsize=(12, 4.4), constrained_layout=True)
axes[0].imshow(source_rgb)
axes[0].add_patch(Rectangle((sample_meta.x1, sample_meta.y1), sample_meta.w, sample_meta.h,
                            fill=False, edgecolor=PALETTE["accent"], linewidth=2))
axes[0].set_title("Source photo and bounding box", fontsize=10.5)
panels = [(axes[1], eval_image, eval_known, eval_positives, "Evaluation input: crop, resize, normalize"),
          (axes[2], aug_image, aug_known, aug_positives, f"Training input: flip and rotation (seed {AUG_SEED})")]
for ax, image, known, positives, title in panels:
    ax.imshow(to_rgb(image))
    overlay = ax.imshow(np.ma.masked_where(~known, positives), cmap=cmap, vmin=1, vmax=vmax, alpha=0.65,
                        extent=(-0.5, 223.5, 223.5, -0.5), interpolation="nearest")
    for k in range(1, 14):  # the 14 x 14 grid of 16-pixel patches the spatial concepts live on
        ax.axhline(16 * k - 0.5, color="white", linewidth=0.4, alpha=0.6)
        ax.axvline(16 * k - 0.5, color="white", linewidth=0.4, alpha=0.6)
    ax.set_title(title, fontsize=10.5)
for ax in axes:
    ax.axis("off")
fig.colorbar(overlay, ax=list(axes[1:]), shrink=0.75, ticks=MaxNLocator(integer=True),
             label="Positive concept targets per patch")
fig.suptitle("Concept targets are built after augmentation, so they stay on the bird")
plt.show()

attribute_names = pd.read_csv(DATASET_DIR / "attributes.txt", sep=" ", names=["attr_id", "definition"]).set_index("attr_id").definition
global_present = [attribute_names[train_set.non_spatial_attributes_pos[i]].replace("has_", "").replace("::", ": ").replace("_", " ")
                  for i in np.flatnonzero(eval_global.numpy() == 1)]
species = train_set.class_names[int(class_target)].split(".", 1)[-1].replace("_", " ")
caption(f"<b>Figure 3.1</b> {species}, training image {SAMPLE_INDEX}. Shaded cells are patches with spatial "
        f"concept targets: {int(eval_known.sum())} of {eval_known.size} in the evaluation input, "
        f"{int(aug_known.sum())} after augmentation; colour counts the positive concepts per patch. "
        "An unshaded patch has no target (NaN), which is not the same as a negative one. "
        f"Global concepts annotated present: {', '.join(global_present) or 'none'}.")

<div style="border-top: 3px solid #ac543f; padding-top: 8px;"><span style="font-size: 11px; letter-spacing: 0.12em; text-transform: uppercase; font-weight: 600;">Live demo</span><span style="font-size: 13.5px; opacity: 0.7;"> &middot; change <code>BATCH_SIZE</code> and re-run</span></div>

In [ ]:
# Live: one real batch exactly as the model receives it (evaluation transform, no worker processes).
BATCH_SIZE = 2  # images per batch; training used 32

import torch
from torch.utils.data import DataLoader
from data import CUB2011Parts, CUB2011Parts_dataset

torch.set_num_threads(CPU_THREADS)
assert (DATA_DIR / "CUB_200_2011" / "cropped").is_dir(), "No crop cache: the dataset would start writing crops. See 3.4."
batch_module = CUB2011Parts(data_dir=str(DATA_DIR), batch_size=BATCH_SIZE, num_workers=0)
batch_set = CUB2011Parts_dataset(root=str(DATA_DIR), train=True, crop=True, download=False,
                                 transform=batch_module.test_transform)
images_batch, global_batch, spatial_batch, labels_batch = next(iter(
    DataLoader(batch_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)))
assert images_batch.shape[1:] == (3, 224, 224) and global_batch.shape[1:] == (13,)
assert spatial_batch.shape[1:] == (196, 95) and labels_batch.dtype == torch.int64
fact_table([
    (name, f"{tuple(tensor.shape)} · {str(tensor.dtype).replace('torch.', '')} · "
           f"{int(torch.isfinite(tensor).sum()):,} of {tensor.numel():,} entries known")
    for name, tensor in [("Images", images_batch), ("Global concept targets", global_batch),
                         ("Spatial concept targets", spatial_batch), ("Class labels", labels_batch)]
], title=f"One batch of {BATCH_SIZE}, as passed to the model; unknown entries are NaN")

### 3.4 Data issues

| Issue | Origin | Handling |
| --- | --- | --- |
| `attributes.txt` extracts one folder above where the loader [reads it](https://github.com/IBM/concept_transformer/blob/84c9c30108a6249e017d580c4120ea54af0064f7/data/cub2011parts.py#L176-L179) | Archive layout vs. repository loader | Copy it inward, keep the original, check the bytes match |
| A partial crop cache goes unnoticed | Repository: cropping is [skipped whenever `cropped/` exists](https://github.com/IBM/concept_transformer/blob/84c9c30108a6249e017d580c4120ea54af0064f7/data/cub2011parts.py#L76-L80) | Check every expected crop path before training |
| Some attribute rows carry a surplus trailing field | Official annotations | Keep the file; the pinned pandas still reads the ID and presence fields the model uses |
| A few bounding boxes extend past their image | Official annotations | Keep the boxes; the repository's Pillow crop pads them |

<div style="border-top: 3px solid #ac543f; padding-top: 8px;"><span style="font-size: 11px; letter-spacing: 0.12em; text-transform: uppercase; font-weight: 600;">Live demo</span><span style="font-size: 13.5px; opacity: 0.7;"> &middot; re-checks the files on disk, read-only</span></div>

In [ ]:
# Live: re-check each data issue against the files on disk (read-only), beside the saved audit counts.
import hashlib

DATASET_DIR = DATA_DIR / "CUB_200_2011"
source_audit = json.loads((SETUP_LOGS / "cub-download.json").read_text())
crop_audit = json.loads((SETUP_LOGS / "cub-cropping.json").read_text())


def unchanged(relative_path):
    """True when an annotation file still matches the SHA-256 recorded by the source audit."""
    digest = hashlib.sha256((DATASET_DIR / relative_path).read_bytes()).hexdigest()
    return digest == source_audit["annotation_sha256"][relative_path]


outer_attributes, inner_attributes = DATA_DIR / "attributes.txt", DATASET_DIR / "attributes.txt"
attributes_ok = (inner_attributes.is_file() and outer_attributes.is_file()
                 and inner_attributes.read_bytes() == outer_attributes.read_bytes())
image_paths = pd.read_csv(DATASET_DIR / "images.txt", sep=" ", names=["image_id", "path"]).path
crops_present = sum((DATASET_DIR / "cropped" / path).is_file() for path in image_paths)
surplus_rows = source_audit["attributes"]["source_rows_with_surplus_zero_token"]
surplus_images = sorted(source_audit["attributes"]["surplus_token_image_id_counts"], key=int)
boxes = source_audit["images"]
fact_table([
    ("attributes.txt in the loader's folder",
     "present and byte-identical to the archive copy" if attributes_ok else "MISSING or different: copy it before training"),
    ("Crop cache",
     f"{crops_present:,} of {len(image_paths):,} expected crops on disk now; the full crop audit decoded "
     f"{crop_audit['fully_decoded_count']:,}" + (" in the source copy" if Path(crop_audit["data_root"]) != DATA_DIR else "")),
    ("Attribute rows with a surplus field",
     f"{surplus_rows:,} rows, all in images {' and '.join(surplus_images)}; file "
     + ("unchanged since the audit" if unchanged("attributes/image_attribute_labels.txt") else "CHANGED since the audit")),
    ("Boxes extending past their image",
     f"{boxes['bounding_boxes_beyond_image_plus_one_pixel']} of {boxes['bounding_boxes_positive_finite']:,}, "
     "by more than one pixel; file "
     + ("unchanged since the audit" if unchanged("bounding_boxes.txt") else "CHANGED since the audit")),
], title="Data issues, checked now")

### 3.5 Validation split

<div style="border-top: 3px solid #ac543f; background: rgba(172, 84, 63, 0.07); padding: 12px 18px 14px 18px;"><div style="font-size: 11px; letter-spacing: 0.12em; text-transform: uppercase; opacity: 0.65; margin-bottom: 6px;">Key point</div><div style="font-family: Newsreader, Georgia, serif; font-size: 21px; line-height: 1.4;">The repository chooses its checkpoint on images it later tests on.</div></div>

Its `setup()` [draws the validation set at random from the official test split](https://github.com/IBM/concept_transformer/blob/84c9c30108a6249e017d580c4120ea54af0064f7/data/cub2011parts_datamodule.py#L38-L51), then tests on the whole split. The paper describes no validation split for CUB. This is naturally problematic becasue it chooses a checkpoint that favours the test set. So we kept this protocol as the like-for-like run and trained a second run on a disjoint split, with every other setting unchanged.

<div style="border-top: 3px solid #ac543f; padding-top: 8px;"><span style="font-size: 11px; letter-spacing: 0.12em; text-transform: uppercase; font-weight: 600;">Live demo</span><span style="font-size: 13.5px; opacity: 0.7;"> &middot; runs the repository's own <code>setup()</code> &middot; change <code>SEED</code>: the overlap stays the same</span></div>

In [ ]:
# Live: run the repository's CUB2011Parts.setup() and count validation images that are also test images.
SEED = 0  # torch seed for random_split; every seed gives the same overlap
UPSTREAM_SHA256 = "b6f1b6f3239718e445b167a94a52395693f3ac3e80786548023a7e86d25021d0"  # data/cub2011parts_datamodule.py at commit 84c9c30

import hashlib
import inspect
import torch
from data import CUB2011Parts

torch.set_num_threads(CPU_THREADS)
assert (DATA_DIR / "CUB_200_2011" / "cropped").is_dir(), "No crop cache: the dataset would start writing crops. See 3.4."
torch.manual_seed(SEED)
repo_module = CUB2011Parts(data_dir=str(DATA_DIR), batch_size=32, num_workers=0)
repo_module.setup()  # stage None builds the training, validation and test sets
val_subset = repo_module.cub_val
val_ids = set(val_subset.dataset.data.img_id.iloc[val_subset.indices])
test_ids = set(repo_module.cub_test.data.img_id)
train_ids = set(repo_module.cub_train.data.img_id)
stat_row([
    {"label": "Validation images", "value": f"{len(val_ids):,}",
     "detail": "drawn by random_split from the official test split", "color": "neg"},
    {"label": "Also in the final test set", "value": f"{len(val_ids & test_ids):,}",
     "detail": f"the test set is all {len(test_ids):,} official test images", "color": "warn"},
    {"label": "Shared with training", "value": f"{len(val_ids & train_ids):,}",
     "detail": "training images stay separate", "color": "pos"},
])
module_file = Path(inspect.getsourcefile(CUB2011Parts))
pinned = hashlib.sha256(module_file.read_bytes()).hexdigest() == UPSTREAM_SHA256
split_line = next((line.strip() for line in inspect.getsource(CUB2011Parts.setup).splitlines()
                   if "random_split(" in line), "no random_split call found")
print(f"{module_file.name} is {'identical to' if pinned else 'DIFFERENT from'} the pinned upstream file.")
print("CUB2011Parts.setup():", split_line)

In [ ]:
# Evidence: where each completed run drew its images, read from the saved run records.
official = json.loads((SETUP_LOGS / "cub-download.json").read_text())["split_counts"]  # official train / test pools
repo_sizes = RUNS[REPO_RUN]["snapshot"]["dataset_sizes"]
split = RUNS[DISJOINT_RUN]["run"]["split"]
disjoint_sizes, from_pool = split["counts"], split["validation_origin_counts"]
assert repo_sizes["train"] == official["train"] and repo_sizes["test"] == official["test"]
assert str(repo_sizes["validation"]) in RUNS[REPO_RUN]["snapshot"]["split_limitation"]  # "overlap: N image IDs"
assert disjoint_sizes == RUNS[DISJOINT_RUN]["snapshot"]["dataset_sizes"]
assert disjoint_sizes["train"] + from_pool["official_train"] == official["train"]
assert disjoint_sizes["test"] + from_pool["official_test"] == official["test"]
assert split["manifest_sha256"] == EVIDENCE["disjoint_training_job"]["manifest_sha256"]
total = official["train"] + official["test"]
assert sum(disjoint_sizes.values()) == total
ROLE_ALPHA = {"train": 0.30, "validation": 1.0, "test": 0.60}  # solid = validation, the set that picks the checkpoint
rows = [
    (1, REPO_RUN, [(0, official["train"], "train"), (official["train"], official["test"], "test")]),
    (0, DISJOINT_RUN, [(0, disjoint_sizes["train"], "train"),
                       (disjoint_sizes["train"], disjoint_sizes["validation"], "validation"),
                       (official["train"] + from_pool["official_test"], disjoint_sizes["test"], "test")]),
]
fig, ax = plt.subplots(figsize=(9, 3.4), constrained_layout=True)
for y, job, segments in rows:
    for start, width, role in segments:
        ax.barh(y, width, left=start, height=0.5, color=RUN_COLOR[job], alpha=ROLE_ALPHA[role],
                edgecolor="white", linewidth=1)
        if role != "validation":
            ax.text(start + width / 2, y, f"{role}  {width:,}", ha="center", va="center", fontsize=10)
ax.barh(1, repo_sizes["validation"], left=official["train"], height=0.24, color=RUN_COLOR[REPO_RUN],
        edgecolor=PALETTE["warn"], linewidth=1.5)  # the repository's validation set sits inside its test set
ax.text(official["train"] + 80, 1.31, f"validation {repo_sizes['validation']:,}, drawn from these test images",
        ha="left", va="bottom", fontsize=9)
ax.text(disjoint_sizes["train"], -0.31,
        f"validation {disjoint_sizes['validation']:,} = {from_pool['official_train']} training + "
        f"{from_pool['official_test']} test images", ha="left", va="top", fontsize=9)
ax.axvline(official["train"], color="0.35", linestyle=":", linewidth=1)
ax.text(official["train"] - 80, 1.8, "official training images", ha="right", va="bottom", fontsize=9, alpha=0.75)
ax.text(official["train"] + 80, 1.8, "official test images", ha="left", va="bottom", fontsize=9, alpha=0.75)
ax.set_yticks([0, 1])
ax.set_yticklabels([f"{RUN_LABEL[job]}\njob {job}" for job in (DISJOINT_RUN, REPO_RUN)])
ax.set(xlim=(0, total), ylim=(-0.65, 2.0), xlabel=f"Images (all {total:,} in CUB-200-2011)",
       title="Repository validation reuses test images; the disjoint split keeps all three sets apart")
ax.grid(False, axis="y")
ax.set_axisbelow(True)
plt.show()
caption(f"<b>Figure 3.2</b> Where each completed run's images come from. Repository split: its "
        f"{repo_sizes['validation']:,} validation images are drawn from the {official['test']:,} official test "
        "images, which are all tested again. Disjoint split: pairwise overlaps are "
        + ", ".join(f"{pair.replace('_', '–')} {count}" for pair, count in split["pairwise_overlap_counts"].items())
        + f"; its image IDs are fixed in <code>split_manifest.json</code> (SHA-256 {split['manifest_sha256'][:12]}…, "
        f"split seed {split['split_seed']}).")

<details><summary style="cursor: pointer; font-size: 15px;"><b>Backup:</b> how the disjoint split was built</summary>

- Keep the repository's train : validation : test proportions, renormalized over unique images (the repository counts its validation images twice), with largest-remainder rounding.
- Reserve the validation images per class from both official pools. Training then uses only official training images, and the final test only official test images.
- Generate the split once with a fixed seed and save the image IDs in `split_manifest.json`. The training run and the explanation script read that manifest instead of re-splitting inside `setup()`.
- Model, losses, optimizer, schedule, preprocessing and hyperparameters match the repository run. The checkpoint is chosen on validation cross-entropy and tested once.

</details>